# ImamAli110 — Interim Production Test 001 (GPU unavailable)

این نوت‌بوک مرحله‌های مستقل از GPU را آماده می‌کند: داستان فارسی، قلاب آغازین، تقسیم‌بندی صحنه‌ها، نریشن فارسی و فایل زیرنویس. **مدل تولید تصویر SDXL را اجرا نمی‌کند و چیزی در یوتیوب آپلود نمی‌کند.** هدف این است تا زمان آزادشدن GPU، دارایی‌های قابل استفاده آماده شوند.

اگر TPU متصل باشد، آن را فقط تشخیص می‌دهد؛ این مرحله‌ها عمدتاً شبکه/CPU هستند و TPU الزاماً سرعتشان را بیشتر نمی‌کند.

In [ ]:
# 1) Check runtime and prepare workspace
import os, sys, json, subprocess, importlib.util
from pathlib import Path
print('Python:', sys.version)
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
except Exception as e:
    print('PyTorch check:', repr(e))
try:
    import torch_xla
    print('torch_xla available: yes')
except Exception:
    print('torch_xla available: no (not needed for story/TTS preparation)')

WORK = Path('/content/ImamAli110_interim_001')
WORK.mkdir(parents=True, exist_ok=True)
print('Workspace:', WORK)
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq ffmpeg fonts-noto-core fontconfig'], check=True)
subprocess.run(['fc-cache','-f'], check=False)
subprocess.check_call([sys.executable,'-m','pip','install','-q','edge-tts'])
print('Preparation dependencies ready.')

In [ ]:
# 2) Create one non-repetitive Persian story with a clear opening hook
story = {
  'id': 'imamali110_interim_001',
  'title_fa': 'درختی که برای غریبه‌ها کاشته شد',
  'hook_fa': 'اگر امروز کسی از خوبی تو سود نبرد، یعنی خوبی‌ات بی‌فایده بوده؟',
  'voice': 'fa-IR-FaridNeural',
  'rate': '-8%',
  'beats': [
    {'scene': 1, 'seconds_hint': 5, 'narration': 'اگر امروز کسی از خوبی تو سود نبرد، یعنی خوبی‌ات بی‌فایده بوده؟', 'visual_prompt': 'Cinematic close-up of a tiny green sapling in dry soil at sunrise, no people, hopeful realistic photography.'},
    {'scene': 2, 'seconds_hint': 5, 'narration': 'روزی کودکی نهالی کاشت؛ با اینکه می‌دانست شاید سال‌ها طول بکشد تا سایه بدهد.', 'visual_prompt': 'A young Middle Eastern boy carefully planting a small sapling in a quiet village garden, single child, natural proportions, golden morning light.'},
    {'scene': 3, 'seconds_hint': 5, 'narration': 'مردی از او پرسید: وقتی شاید خودت زیر سایه‌اش ننشینی، چرا زحمت می‌کشی؟', 'visual_prompt': 'An elderly Middle Eastern man speaking kindly with a young boy beside a sapling, exactly two people, clear separation, realistic anatomy.'},
    {'scene': 4, 'seconds_hint': 5, 'narration': 'کودک گفت: من امروز زیر سایه‌ی درختی نشسته‌ام که کسی پیش از من کاشته بود.', 'visual_prompt': 'The boy looking up with quiet sincerity beneath the edge of a mature oak tree, one person only, natural child proportions.'},
    {'scene': 5, 'seconds_hint': 5, 'narration': 'بعضی کارهای خوب، جوابشان را از آینده می‌گیرند؛ نه از تشویق امروز.', 'visual_prompt': 'Warm sunlight filtering through the leaves of a mature tree, peaceful empty bench, cinematic realistic photography, no text.'},
    {'scene': 6, 'seconds_hint': 5, 'narration': 'پس خوبی کن؛ شاید روزی غریبه‌ای به لطفی تکیه کند که امروز تو بی‌صدا انجام دادی.', 'visual_prompt': 'A boy and an elderly man watering a small sapling together, exactly two people, hopeful golden-hour documentary photography.'}
  ],
  'cta_fa': 'اگر این پیام به دلت نشست، برای کسی بفرست که به امید نیاز دارد.',
  'safety_note': 'داستان تمثیلی است؛ به عنوان نقل‌قول تاریخی یا دینی معرفی نشود.'
}
assert story['hook_fa'] == story['beats'][0]['narration']
assert len(story['beats']) == 6 and all(len(b['narration']) > 15 for b in story['beats'])
(WORK/'storyboard.json').write_text(json.dumps(story, ensure_ascii=False, indent=2), encoding='utf-8')
print('Story QC passed:', len(story['beats']), 'scenes; hook is first narration line.')
print(story['hook_fa'])

In [ ]:
# 3) Generate Persian narration segments (no GPU required)
import asyncio, edge_tts
audio_dir = WORK/'audio'
audio_dir.mkdir(exist_ok=True)
async def make_audio():
    for beat in story['beats']:
        target = audio_dir/f"scene_{beat['scene']:02d}.mp3"
        await edge_tts.Communicate(beat['narration'], story['voice'], rate=story['rate'], pitch='-1Hz').save(str(target))
        print('Created', target.name, target.stat().st_size, 'bytes')
await make_audio()
assert len(list(audio_dir.glob('scene_*.mp3'))) == len(story['beats'])
print('TTS QC passed: all scene audio files exist.')

In [ ]:
# 4) Make a readable UTF-8 SRT draft and concatenate voice segments for listening
def srt_time(seconds):
    ms = int(round(seconds*1000)); h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); s, ms = divmod(ms, 1000)
    return f'{h:02}:{m:02}:{s:02},{ms:03}'
lines=[]; t=0.0
for i, beat in enumerate(story['beats'], 1):
    # Rough display timing only; audio durations are measured below and should replace these before final edit.
    d=float(beat['seconds_hint'])
    lines += [str(i), f'{srt_time(t)} --> {srt_time(t+d)}', beat['narration'], '']
    t += d
(WORK/'captions_draft.srt').write_text('\n'.join(lines), encoding='utf-8')
concat = WORK/'concat.txt'
concat.write_text(''.join(f"file '{(audio_dir/f'scene_{b['scene']:02d}.mp3').as_posix()}'\n" for b in story['beats']), encoding='utf-8')
preview = WORK/'narration_preview.mp3'
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',str(concat),'-c:a','libmp3lame','-q:a','3',str(preview)], check=True, capture_output=True)
print('Created:', preview)
print('Created:', WORK/'captions_draft.srt')
print('Created:', WORK/'storyboard.json')

In [ ]:
# 5) Download the generated files to review; no upload/publishing occurs
from google.colab import files
import shutil
bundle = WORK/'ImamAli110_interim_001.zip'
shutil.make_archive(str(bundle.with_suffix('')), 'zip', WORK)
print('Bundle ready:', bundle)
files.download(str(bundle))